# duck: condensed SWOT SLC export
Download this notebook and run all cells in Jupyter with Python 3.10+. It installs missing dependencies and prompts for your own [NASA Earthdata credentials](https://urs.earthdata.nasa.gov/) if needed. Internet access, package-install permission, and a few GB of RAM are required. No companion files or repository are needed.

Produces a condensed pickle and radar/satellite context PNGs. The pickle contains both original-resolution complex SLC channels and a small metadata dictionary. Plots are preview-only; no full-context pickle is generated. Outputs go under `swot_slc_exports/duck/` in the working directory; change `OUTPUT_BASE` below to choose another location. Only load trusted pickle files.


In [ ]:
# Install missing packages into this notebook's Python environment.
import importlib.util, subprocess, sys
packages={'matplotlib':'matplotlib','contextily':'contextily','numpy':'numpy','xarray':'xarray','earthaccess':'earthaccess',
          'h5netcdf':'h5netcdf','h5py':'h5py','pyproj':'pyproj'}
missing=[package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Dependencies ready. Python:',sys.version.split()[0])


In [ ]:
SITE='duck'
DATE='2023-09-13'
CYCLE='003'
PASS='354'
TILE='091L'

from pathlib import Path
OUTPUT_BASE = Path.cwd() / 'swot_slc_exports'


In [ ]:
from pathlib import Path
import json, pickle, re
import numpy as np
import xarray as xr
import earthaccess
OUT = OUTPUT_BASE / 'duck'
OUT.mkdir(parents=True, exist_ok=True)
print('Outputs:',OUT.resolve())
auth = earthaccess.login(persist=False)
if not auth.authenticated: raise RuntimeError('Earthdata login failed; check credentials and product access agreements')
def find(product):
    results = earthaccess.search_data(short_name=product, temporal=(DATE+'T00:00:00Z', DATE+'T23:59:59Z'), granule_name=f'*_{CYCLE}_{PASS}_{TILE}_*')
    if len(results)!=1: raise RuntimeError(f'Expected one {product} granule, found {len(results)}')
    return results[0]
slc_g, pixc_g = find('SWOT_L1B_HR_SLC_D'), find('SWOT_L2_HR_PIXC_D')
handles = earthaccess.open([slc_g])
phandles = earthaccess.open([pixc_g])
slc = xr.open_dataset(handles[0], group='slc', engine='h5netcdf')
pr = xr.open_dataset(phandles[0], engine='h5netcdf')
pc = xr.open_dataset(phandles[0], group='pixel_cloud', engine='h5netcdf')
attrs = {**pr.attrs, **pc.attrs}
lon, lat = pc.longitude.values, pc.latitude.values
lon = (lon+180)%360-180


In [ ]:
from pyproj import Transformer
# Frozen from the original FRF overview, using its survey-derived UTM 18N -> FRF affine fit.
# Used only to select pixels: exported channels retain their original radar coordinates.
xlim=(-820.0,17046.062041999998)
ylim=(-75.36791900000003,5045.931172)
coef=np.array([[0.9529582553040513, -0.30425142345801165], [0.30425142392441196, 0.9529582538026785], [-1630428.919140883, -3683943.03847706]],dtype=float)
trans=Transformer.from_crs(4326,32618,always_xy=True)
e,north=trans.transform(lon,lat)
xy=np.column_stack([e,north,np.ones_like(e)])@coef
selected=(xy[:,0]>=xlim[0])&(xy[:,0]<=xlim[1])&(xy[:,1]>=ylim[0])&(xy[:,1]<=ylim[1])
if not selected.any(): raise RuntimeError('Requested Duck overview has no PIXC coverage')
selection=dict(frf_x_m=xlim,frf_y_m=ylim,geographic_bounds_of_selected_points=[float(np.nanmin(lon[selected])),float(np.nanmin(lat[selected])),float(np.nanmax(lon[selected])),float(np.nanmax(lat[selected]))])


In [ ]:
az = pc.azimuth_index.values
rg = pc.range_index.values
valid = selected & np.isfinite(az) & np.isfinite(rg)
if not valid.any(): raise RuntimeError('Requested region has no PIXC coverage')
# PIXC rare-grid indices map to native SLC lines using product offsets/looks.
looks = float(attrs['num_azimuth_looks'])
if looks != round(looks) or looks < 1: raise ValueError('Invalid azimuth look count')
looks = int(looks)
offset = int(attrs['azimuth_offset'])
r0 = offset + int(az[valid].min())*looks
r1 = offset + (int(az[valid].max())+1)*looks
c0, c1 = int(rg[valid].min()), int(rg[valid].max())+1
ad, rd = slc.slc_plus_y.dims[:2]
if not (0 <= r0 < r1 <= slc.sizes[ad] and 0 <= c0 < c1 <= slc.sizes[rd]):
    raise ValueError('PIXC mapping falls outside SLC: inspect product offsets before proceeding')
print('Native crop:', (r0,r1,c0,c1), 'shape:', (r1-r0,c1-c0))
print('Two-channel estimated size (GB):', (r1-r0)*(c1-c0)*16/1e9)
subset = slc.isel({ad:slice(r0,r1), rd:slice(c0,c1)}).load()
def complex_channel(a):
    v = a.values
    if np.iscomplexobj(v): return v
    if v.ndim!=3 or v.shape[-1]!=2: raise ValueError('Unexpected complex encoding')
    return v[...,0] + 1j*v[...,1]
s1, s2 = complex_channel(subset.slc_plus_y), complex_channel(subset.slc_minus_y)
metadata = dict(site=SITE, date=DATE, cycle=CYCLE, pass_number=PASS, tile=TILE,
    granule=slc_g['umm']['GranuleUR'], temporal=slc_g['umm']['TemporalExtent'],
    selection=selection, native_crop_half_open=[r0,r1,c0,c1],
    convention='Original SLC azimuth rows, range columns; no resampling',
    geolocation='Supporting PIXC coordinates are not native per-pixel SLC geolocation')


## Preview plots
Radar amplitude and raw interferometric phase use every second sample for display. Amplitude contrast uses PIXC open-water points where available. Satellite context shows PIXC WSE with a color bar; these supporting data are not added to the condensed pickle.

In [ ]:
import matplotlib.pyplot as plt
# Only display is thinned; exported channels retain all samples.
step = 2
a,b = s1[::step,::step],s2[::step,::step]
fig, axes = plt.subplots(2,1,figsize=(12,9),layout='constrained')
amp=np.abs(a)
# Use PIXC open-water classification to select contrast samples on the native SLC grid.
water=valid & (pc.classification.values==4)
if 'bright_land_flag' in pc:
    water &= ~np.isin(pc.bright_land_flag.values,[1,2])
wa=az[water].astype('int64'); wr=rg[water].astype('int64')
# Use the center raw line of each PIXC look block; do not interpolate amplitudes.
sample_rows=offset+wa*looks+looks//2
inside=(sample_rows>=r0)&(sample_rows<r1)&(wr>=c0)&(wr<c1)
water_amplitude=np.abs(s1[sample_rows[inside]-r0,wr[inside]-c0])
water_amplitude=water_amplitude[np.isfinite(water_amplitude)]
if water_amplitude.size<100: raise ValueError('Too few PIXC water samples for reliable amplitude limits')
lo,hi=np.percentile(water_amplitude,[1,99])
print('Water-only amplitude limits:',lo,hi,'from',water_amplitude.size,'PIXC-mapped samples',flush=True)

for ax,image,cmap,limits,title in zip(axes,[amp,np.angle(a*np.conj(b))],['gray','twilight'],[(lo,hi),(-np.pi,np.pi)],['plus_y amplitude','Raw phase: plus_y × conjugate(minus_y)']):
    im=ax.imshow(image,origin='lower',aspect='auto',extent=[c0,c1,r0,r1],cmap=cmap,vmin=limits[0],vmax=limits[1],interpolation='nearest')
    ax.set(xlabel='Original range index',ylabel='Original azimuth index',title=f'{SITE} {DATE}: {title}')
    fig.colorbar(im,ax=ax)
fig.savefig(OUT/f'{SITE}_{DATE}_condensed_radar.png',dpi=160)
plt.show()
# Satellite context with tide-corrected, geoid-referenced PIXC WSE.
import contextily as cx
context=pc.isel(points=np.flatnonzero(valid))
wse=(context.height-context.geoid-context.solid_earth_tide-context.load_tide_fes-context.pole_tide).values
clon=((context.longitude.values+180)%360)-180
clat=context.latitude.values
good=np.isfinite(wse)&np.isfinite(clon)&np.isfinite(clat)
# Context-only science filtering; raw SLC exports are unchanged.
if 'classification' in context:
    good &= ~np.isin(context.classification.values,[1,2,3,5,6,7])
if 'bright_land_flag' in context:
    good &= ~np.isin(context.bright_land_flag.values,[1,2])
water_only_good=good.copy()
for key in ['sig0_qual','interferogram_qual','geolocation_qual']:
    if key not in context: continue
    flag=context[key]
    bad=0
    for mask,meaning in zip(flag.attrs['flag_masks'],flag.attrs['flag_meanings'].split()):
        if any(word in meaning for word in ['bad','degraded','suspect','phase_unwrap']): bad |= int(mask)
    values=flag.fillna(-1).values.astype('int64')
    good &= (values & bad)==0
context_note=''
if not good.any() and water_only_good.any():
    good=water_only_good
    context_note=' (quality checks relaxed: no qualifying water points)'
if not good.any():
    good=np.isfinite(wse)&np.isfinite(clon)&np.isfinite(clat)
    context_note=' (unfiltered PIXC: no qualifying water points)'
if not good.any(): raise ValueError('No finite WSE in selected region')
lo,hi=np.nanpercentile(wse[good],[5,95])
fig,ax=plt.subplots(figsize=(11,8))
sc=ax.scatter(clon[good],clat[good],c=wse[good],s=1,alpha=0.3,cmap='viridis',vmin=lo,vmax=hi,zorder=2)
ax.set_xlim(float(np.min(clon[good])),float(np.max(clon[good])))
ax.set_ylim(float(np.min(clat[good])),float(np.max(clat[good])))
cx.add_basemap(ax,crs='EPSG:4326',source=cx.providers.Esri.WorldImagery,zorder=0)
fig.colorbar(sc,ax=ax,label='WSE (m; geoid referenced, tide corrected)')
ax.set(xlabel='Longitude',ylabel='Latitude',title=f'{SITE} {DATE}: PIXC WSE over satellite imagery{context_note}')
fig.savefig(OUT/f'{SITE}_{DATE}_condensed_region.png',dpi=180)
plt.show()


In [ ]:
# Save only the native complex arrays and small acquisition/crop metadata.
metadata['interferogram_convention']='slc_plus_y * conjugate(slc_minus_y)'
metadata['array_axes']=['azimuth_row','range_column']
condensed=dict(metadata=metadata,slc_plus_y=np.ascontiguousarray(s1),slc_minus_y=np.ascontiguousarray(s2))
path=OUT/f'{SITE}_{DATE}_slc_condensed.pkl'
with path.with_suffix('.pkl.tmp').open('wb') as f:
    pickle.dump(condensed,f,protocol=5)
if SITE=='boston_harbor' and path.with_suffix('.pkl.tmp').stat().st_size>=35_000_000:
    raise RuntimeError('Export exceeds 35 MB; reduce the geographic bounds')
path.with_suffix('.pkl.tmp').replace(path)
# All exported values are in-memory arrays/scalars, with no remote file handles.
for ds in [subset,slc,pc,pr]: ds.close()
for handle in [*handles,*phandles]: handle.close()
print(f'Saved {path.resolve()} ({path.stat().st_size / 1e6:.2f} MB)')


## Open the pickle
This cell can be used on its own. Set `PICKLE_PATH` to your downloaded pickle. Only Python and NumPy are needed.

In [ ]:
# Open the condensed pickle (NumPy only; no Earthdata login is needed).
# When using this cell independently, replace PICKLE_PATH with your file location.
from pathlib import Path
import pickle
import numpy as np
PICKLE_PATH = Path('swot_slc_exports') / 'duck' / 'duck_2023-09-13_slc_condensed.pkl'
with Path(PICKLE_PATH).open('rb') as f:
    data=pickle.load(f)
slc_plus_y=data['slc_plus_y']
slc_minus_y=data['slc_minus_y']
metadata=data['metadata']
assert slc_plus_y.shape==slc_minus_y.shape
assert slc_plus_y.dtype==slc_minus_y.dtype==np.complex64
print('Channel shape:',slc_plus_y.shape,'dtype:',slc_plus_y.dtype)
